# 00. От Claude Code к OpenClaw

Как coding agent превратился в постоянно доступного персонального агента — и как собрать его минимальный аналог.

**Вопрос аудитории:** чем агент отличается от обычного чата с LLM?


## Путь, который мы сегодня пройдём

> Claude Code показал, как агент может работать внутри инженерного окружения.  
> OpenClaw вынес похожий пользовательский и архитектурный паттерн за пределы терминала — в постоянного персонального агента, доступного из мессенджера.  
> Но вместе с удобством появились архитектурные и эксплуатационные проблемы.  
> На workshop мы соберём упрощённый аналог и разберём, из каких частей он реально состоит.

Это не утверждение, что OpenClaw является форком или прямым техническим продолжением Claude Code. Мы сравниваем эволюцию интерфейса и agent-runtime паттерна.


## Визуальное ядро: три контракта

![Agent system evolution](../visuals/openclaw_path/06_agent_evolution_core.svg)

```text
LLM:         prompt → response
Claude Code: goal → agent loop → tools → repository
OpenClaw:    message → gateway → agent loop → tools → external world
```


# 1. До агентов: LLM как функция

```text
prompt → LLM → text
```

Модель ничего не знает о текущем проекте, не читает файлы, не запускает команды, не хранит рабочее состояние и не продолжает цепочку действий сама.

> Большая языковая модель сама по себе не агент. Агент появляется тогда, когда модель помещают в цикл с состоянием, инструментами и средой выполнения.


## Agent harness

**Agent harness** — это обвязка вокруг модели:

```text
model
+ loop
+ state
+ tools
+ context policy
+ execution policy
+ observability
```

В `01` мы начнём собирать такой harness через `create_deep_agent()`: сначала без рабочей директории, затем с filesystem backend.


# 2. Claude Code: агент внутри рабочей директории

Claude Code — понятный референс coding agent. В официальном overview он описан как agentic coding tool, который читает codebase, редактирует файлы, запускает команды и интегрируется с developer tools.

Источник: [Claude Code overview](https://code.claude.com/docs/en/overview)


## Claude Code как наблюдаемый контракт

```text
Agent loop
├── Context
│   ├── dialogue
│   ├── CLAUDE.md
│   ├── selected files
│   └── tool results
├── Tools
│   ├── read/search
│   ├── edit/write
│   ├── shell
│   └── MCP/subagents
└── Environment
    └── working directory
```

Claude Code также документирует subagents, permissions, hooks и sandboxing.

Источник: [Claude Code subagents](https://code.claude.com/docs/en/sub-agents)


## Почему Claude Code оказался важен

Обычный чат:

```text
Человек копирует код → получает текст → сам применяет изменения
```

Coding agent:

```text
Человек формулирует цель
→ агент исследует репозиторий
→ выбирает инструменты
→ меняет файлы
→ запускает проверку
→ возвращает результат
```

> Claude Code продал не чат с программистом, а модель, встроенную в рабочий цикл разработчика.


# 3. OpenClaw: агент становится постоянно доступным

OpenClaw описывает себя как personal AI assistant. Gateway daemon выступает control plane для sessions, channels, tools и events.

Источник: [OpenClaw GitHub](https://github.com/openclaw/openclaw)

Главная UX-идея:

```text
написал сообщение с телефона
→ агент что-то сделал
→ ответ пришёл туда же
```


## Канал не равен агенту

![OpenClaw Gateway architecture](../visuals/openclaw_path/07_openclaw_gateway_arch.svg)

```text
incoming message
→ normalization
→ session
→ agent runtime
→ response
→ channel adapter
```

Именно такую схему мы затем воспроизведём через VK bridge и LangGraph.


# 4. Почему OpenClaw одновременно крутой и проблемный

> То, что сделало OpenClaw полезным, одновременно расширило поверхность проблем.

```text
Untrusted input
Excessive capabilities
Untrusted extensions
Observability and control
```


## Что мы решаем в workshop, а что нет

В production эти проблемы требуют policy, isolation, sandboxing, allowlists и approvals.

На workshop мы сосредоточимся на архитектуре и observability:

- каждый tool call виден в LangGraph Studio;
- bridge отделён от agent logic;
- внешние действия вынесены в Python tools;
- SWE-глава показывает проверяемый change/test loop.

OpenClaw security docs обсуждают DM policies, allowlists, context visibility, sandboxing и approval boundaries.

Источник: [OpenClaw security docs](https://docs.openclaw.ai/gateway/security)


## Capabilities и extensions

Когда агент получает shell, файлы, токены, браузер и мессенджеры, ошибка модели перестаёт быть плохим текстом и может стать внешним действием.

OpenClaw security policy говорит, что plugins/extensions загружаются in-process с Gateway и являются trusted code.

Источник: [OpenClaw security policy](https://github.com/openclaw/openclaw/security)

Про supply-chain риск skills: [The Verge, Feb 4 2026](https://www.theverge.com/news/874011/openclaw-ai-skill-clawhub-extensions-security-nightmare)


# 5. Что именно мы построим

> Мы не будем копировать весь продукт. Мы воспроизведём его главный архитектурный путь — от agent loop до messenger-first инженерного агента.


## Workshop architecture

![Workshop architecture](../visuals/openclaw_path/08_workshop_architecture.svg)

```text
VK → Polling bridge → LangGraph API → Deep Agent
                                  ├── filesystem
                                  ├── Jenkins tools
                                  ├── VK tools
                                  └── subagents
```


# 6. Scope: OpenClaw и workshop-аналог

| OpenClaw | Workshop-аналог |
| --- | --- |
| Универсальный персональный агент | Инженерный агент |
| Множество каналов | VK |
| Большая экосистема skills | Контролируемый набор Python tools |
| Собственный Gateway | LangGraph server + bridge |
| Personal automation use cases | Repository и Jenkins workflows |
| Полноценный продукт | Учебный воспроизводимый runtime |

> Наша цель — разобрать OpenClaw на понятные инженерные примитивы и собрать минимальную систему, где каждый слой можно увидеть, запустить и изменить.


## Переход к 01

Начнём с модели, сообщений и цикла выполнения.

Следующая глава:

```text
01_agent_and_filesystem.ipynb
```


## Источники

- [Claude Code overview](https://code.claude.com/docs/en/overview)
- [Claude Code subagents](https://code.claude.com/docs/en/sub-agents)
- [OpenClaw GitHub](https://github.com/openclaw/openclaw)
- [OpenClaw security docs](https://docs.openclaw.ai/gateway/security)
- [OpenClaw security policy](https://github.com/openclaw/openclaw/security)
- [The Verge: OpenClaw skill extensions security story, Feb 4 2026](https://www.theverge.com/news/874011/openclaw-ai-skill-clawhub-extensions-security-nightmare)
